# EoMT Inference — Cityscapes vs COCO


Evaluates two EoMT checkpoints on the Cityscapes validation set:
- **Cityscapes model** — trained for *semantic segmentation* (19 classes)
- **COCO model** — trained for *panoptic segmentation* (133 classes), remapped to the 19 Cityscapes classes for fair comparison

## 1 · Setup

In [ ]:
# Install required dependencies if not already present in the environment

import subprocess, sys
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "pyyaml", "lightning", "torchmetrics", "huggingface_hub", "torchvision"],
    check=True,
)

In [ ]:
DEVICE       = 0
IGNORE_INDEX = 255

In [ ]:
import os, importlib, warnings
import yaml
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import torch
from torch.nn import functional as F
from torch.amp.autocast_mode import autocast
from lightning import seed_everything
from tqdm.auto import tqdm

seed_everything(0, verbose=False)

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

PROJECT_ROOT = (
    "/content/drive/My Drive/"
    "Comprehensive Road Scene Understanding for Autonomous Driving"
)


EOMT_DIR   = f"{PROJECT_ROOT}/MaskArchitectureAnomaly_CourseProject/eomt"
DATA_PATH  = f"{PROJECT_ROOT}/datasets/cityscapes"
MODELS_DIR = f"{PROJECT_ROOT}/MaskArchitectureAnomaly_CourseProject/trained_models"

CS_CKPT   = f"{MODELS_DIR}/eomt_cityscapes.bin"
COCO_CKPT = f"{MODELS_DIR}/eomt_coco.bin"


if EOMT_DIR not in sys.path:
    sys.path.insert(0, EOMT_DIR)

for p in [CS_CKPT, COCO_CKPT, DATA_PATH]:
    print("OK     " if os.path.exists(p) else "MISSING", "—", p)

## 2 · Utility functions

In [ ]:
def _instantiate_from_cfg(cffg, **extra_kwargs):
    """Instantiate any class from a LightningCLI-style config dict."""
    module_name, class_name = cfg["class_path"].rsplit(".", 1)
    cls = getattr(importlib.import_module(module_name), class_name)
    kwargs = {**cfg.get("init_args", {}), **extra_kwargs}
    return cls(**kwargs)


def load_model(config_path, checkpoint_path, num_classes=None, img_size=None, stuff_classes=None):
    """Build and load an EoMT LightningModule from a YAML config and a .bin checkpoint."""
    with open(config_path) as f:
        cfg = yaml.safe_load(f)

    # Override config values when explicitly provided
    data_args  = cfg["data"].get("init_args", {})
    model_args = cfg["model"]["init_args"]

    # Build data module just to resolve img_size / num_classes when not overridden
    if img_size is None or num_classes is None:
        data_module_name, data_class = cfg["data"]["class_path"].rsplit(".", 1)
        dm = getattr(importlib.import_module(data_module_name), data_class)(
            path=DATA_PATH, batch_size=1, num_workers=0,
            check_empty_targets=False, **data_args,
        ).setup()
        img_size    = img_size    or dm.img_size
        num_classes = num_classes or dm.num_classes

    warnings.filterwarnings(
        "ignore",
        message=r".*Attribute 'network' is an instance of `nn\.Module` and is already saved.*",
    )

    encoder_cfg = model_args["network"]["init_args"]["encoder"]
    encoder     = _instantiate_from_cfg(encoder_cfg, img_size=img_size)

    net_cfg    = model_args["network"]
    net_kwargs = {k: v for k, v in net_cfg["init_args"].items() if k != "encoder"}
    network    = _instantiate_from_cfg(
        net_cfg, masked_attn_enabled=False,
        num_classes=num_classes, encoder=encoder, **net_kwargs,
    )

    lit_kwargs = {k: v for k, v in model_args.items() if k != "network"}
    if stuff_classes is not None:
        lit_kwargs["stuff_classes"] = stuff_classes
    elif "stuff_classes" in data_args:
        lit_kwargs["stuff_classes"] = data_args["stuff_classes"]

    model = _instantiate_from_cfg(
        cfg["model"], img_size=img_size,
        num_classes=num_classes, network=network, **lit_kwargs,
    ).eval().to(DEVICE)

    state = torch.load(checkpoint_path, map_location=f"cuda:{DEVICE}", weights_only=True)
    model.load_state_dict(state, strict=False)
    print(f"Loaded {checkpoint_path}  (num_classes={num_classes}, img_size={img_size})")
    return model


# ── Colormap helpers ─────────────────────────────────────────────────────────

def make_colormap(masks, ignore_index):
    """Build a {class_id: RGB} dict from the union of class IDs seen in `masks`."""
    unique = np.unique(np.concatenate([np.unique(m) for m in masks]))
    valid  = unique[unique != ignore_index]
    colors = np.array([plt.cm.hsv(i / len(valid))[:3] for i in range(len(valid))])
    cmap   = {cid: colors[i] for i, cid in enumerate(valid)}
    cmap[ignore_index] = np.array([0., 0., 0.])
    return cmap


def apply_colormap(mask, cmap):
    rgb = np.zeros((*mask.shape, 3))
    for cid in np.unique(mask):
        rgb[mask == cid] = cmap.get(cid, [0., 0., 0.])
    return rgb


def colorize_panoptic(sem, inst):
    """Colorize a panoptic (sem, inst) pair with instance-level borders."""
    ids  = np.unique(sem[sem >= 0])
    cmap = {s: plt.cm.tab20(i / max(len(ids), 1))[:3] for i, s in enumerate(ids)}
    rgb  = np.zeros((*sem.shape, 3))
    for s in ids:
        rgb[sem == s] = cmap[s]
    combined = sem.astype(np.int64) * 10_000 + inst.astype(np.int64)
    border   = np.zeros(sem.shape, bool)
    border[1:,  :]  |= combined[1:,  :] != combined[:-1, :]
    border[:-1, :]  |= combined[1:,  :] != combined[:-1, :]
    border[:,  1:]  |= combined[:,  1:] != combined[:, :-1]
    border[:, :-1]  |= combined[:,  1:] != combined[:, :-1]
    rgb[border] = 0
    return rgb

## 3 · Load Cityscapes data module and models

In [ ]:
CS_CONFIG_PATH   = f"{EOMT_DIR}/configs/dinov2/cityscapes/semantic/eomt_base_640.yaml"
COCO_CONFIG_PATH = f"{EOMT_DIR}/configs/dinov2/coco/panoptic/eomt_base_640_2x.yaml"


# Load data module from the Cityscapes config (used for both models' evaluation)
with open(CS_CONFIG_PATH) as f:
    cs_cfg = yaml.safe_load(f)
data_module_name, data_class = cs_cfg["data"]["class_path"].rsplit(".", 1)
data = getattr(importlib.import_module(data_module_name), data_class)(
    path=DATA_PATH, batch_size=1, num_workers=0, check_empty_targets=False,
    **cs_cfg["data"].get("init_args", {}),
).setup()
val_dataset = data.val_dataloader().dataset

# Cityscapes model — semantic segmentation, 19 classes
cs_model = load_model(CS_CONFIG_PATH, CS_CKPT)

# COCO model — panoptic segmentation, 133 classes
coco_model = load_model(
    COCO_CONFIG_PATH, COCO_CKPT,
    num_classes=133,
    img_size=(640, 640),
    stuff_classes=list(range(80, 133)),
)

## 3b · Load Cityscapes fine-tuned models

In [ ]:
#Load Fine-Tuned Model

import glob
import torch
from peft import LoraConfig, get_peft_model

from models.eomt import EoMT
from models.vit import ViT
from training.mask_classification_semantic import MaskClassificationSemantic

# Configuration
# Supported experiments: "head_only", "blocks_8_11", "lora"
EXPERIMENT = "head_only"

# Determine checkpoint directory based on the selected experiment
if EXPERIMENT == "lora":
    ckpt_dir = f"{EOMT_DIR}/logs/exp_lora/version_0/checkpoints"
elif EXPERIMENT == "blocks_8_11":
    ckpt_dir = f"{EOMT_DIR}/logs/exp_blocks_8_11_v2/version_0/checkpoints"
elif EXPERIMENT == "head_only":
    ckpt_dir = f"{EOMT_DIR}/logs/exp_head_only/version_0/checkpoints"
else:
    raise ValueError(f"Invalid experiment '{EXPERIMENT}'. Choose from: 'head_only', 'blocks_8_11', 'lora'")

# Retrieve the latest checkpoint
ckpt_files = sorted(glob.glob(f"{ckpt_dir}/*.ckpt"))
if not ckpt_files:
    raise FileNotFoundError(f"No checkpoints found in {ckpt_dir}. Please check the path.")

latest_ckpt = ckpt_files[-1]
print(f"Loading {EXPERIMENT} checkpoint from:\n{latest_ckpt}")

# Load state dict
ckpt_data = torch.load(latest_ckpt, map_location="cpu", weights_only=False)
state_dict = ckpt_data["state_dict"]

# Initialize the base encoder
encoder = ViT(backbone_name="vit_base_patch14_reg4_dinov2", img_size=(640, 640))

# Inject LoRA parameters if required
if EXPERIMENT == "lora":
    lora_config = LoraConfig(
        r=8,
        lora_alpha=16,
        target_modules=["qkv"],
        lora_dropout=0.05,
        bias="none",
    )
    encoder = get_peft_model(encoder, lora_config)
    print("PEFT/LoRA wrapper successfully applied to the encoder.")

# Initialize the EoMT network
network = EoMT(encoder=encoder, num_blocks=3, num_q=200, num_classes=19)

# Setup the Lightning module for inference
ft_model = MaskClassificationSemantic(
    network=network,
    img_size=(640, 640),
    num_classes=19,
    attn_mask_annealing_enabled=True,
    attn_mask_annealing_start_steps=[3317, 8292, 13268],
    attn_mask_annealing_end_steps=[6634, 11609, 16585],
    lr=1e-4,
    llrd=0.8,
    llrd_l2_enabled=True,
    lr_mult=1.0,
    weight_decay=0.05,
    poly_power=0.9,
    warmup_steps=[500, 1000],
    load_ckpt_class_head=True,
    ckpt_path=None,
)

# Load weights and set model to evaluation mode
ft_model.load_state_dict(state_dict, strict=True)
ft_model = ft_model.to(DEVICE).eval()

print(f"Model '{EXPERIMENT}' is ready for inference.")

## 4 · Inference functions

In [ ]:
@torch.no_grad()
def run_semantic(model, img):
    """
    Semantic inference (MaskFormer-style argmax).
    Works on *any* EoMT model regardless of training task.
    Per-pixel score: sum_i p_i(c) * sigmoid(m_i[h,w]), then argmax over c.
    """
    with autocast(dtype=torch.float16, device_type="cuda"):
        imgs      = [img.to(DEVICE)]
        img_sizes = [imgs[0].shape[-2:]]
        crops, origins      = model.window_imgs_semantic(imgs)
        ml_per_l, cl_per_l  = model(crops)
        mask_logits = F.interpolate(ml_per_l[-1], model.img_size, mode="bilinear")
        crop_logits = model.to_per_pixel_logits_semantic(mask_logits, cl_per_l[-1])
        logits      = model.revert_window_logits_semantic(crop_logits, origins, img_sizes)
    return logits[0].argmax(0).cpu().numpy()


@torch.no_grad()
def run_panoptic(model, img):
    """
    Panoptic inference — assigns each pixel to the query maximising p_i(c_i)*m_i[h,w].
    Returns (sem_pred, inst_pred) as numpy arrays.
    """
    with autocast(dtype=torch.float16, device_type="cuda"):
        imgs      = [img.to(DEVICE)]
        img_sizes = [imgs[0].shape[-2:]]
        transformed          = model.resize_and_pad_imgs_instance_panoptic(imgs)
        ml_per_l, cl_per_l  = model(transformed)
        mask_logits = F.interpolate(ml_per_l[-1], model.img_size, mode="bilinear")
        mask_logits = model.revert_resize_and_pad_logits_instance_panoptic(mask_logits, img_sizes)
        preds = model.to_per_pixel_preds_panoptic(
            mask_logits, cl_per_l[-1], model.stuff_classes,
            model.mask_thresh, model.overlap_thresh,
        )[0].cpu().numpy()
    return preds[..., 0], preds[..., 1]  # (semantic, instance)

## 5 · COCO → Cityscapes class mapping

In [ ]:
# Panoptic COCO IDs (0-indexed) → Cityscapes train IDs.
# Classes with no semantically valid Cityscapes counterpart are mapped to IGNORE_INDEX (255).
COCO_TO_CS: dict[int, int] = {
    # ── Things (direct matches) ──────────────────────────────────────────────
     0: 11,  # person       → person
     1: 18,  # bicycle      → bicycle
     2: 13,  # car          → car
     3: 17,  # motorcycle   → motorcycle
     5: 15,  # bus          → bus
     6: 16,  # train        → train
     7: 14,  # truck        → truck
     9:  6,  # traffic light→ traffic light
    11:  7,  # stop sign    → traffic sign
    # ── Stuff (categorical merges) ───────────────────────────────────────────
     58:  8,  # potted plant  → vegetation
     88:  8,  # flower        → vegetation
     90:  9,  # gravel        → terrain
     91:  2,  # house         → building
    100:  0,  # road          → road
    102:  9,  # sand          → terrain
    109:  3,  # wall-brick    → wall
    110:  3,  # wall-stone    → wall
    111:  3,  # wall-tile     → wall
    112:  3,  # wall-wood     → wall
    116:  8,  # tree-merged   → vegetation
    117:  4,  # fence-merged  → fence
    119: 10,  # sky-other     → sky
    123:  1,  # pavement      → sidewalk
    125:  8,  # grass-merged  → vegetation
    126:  9,  # dirt-merged   → terrain
    129:  2,  # building-other→ building
    131:  3,  # wall-other    → wall
}

# All remaining COCO IDs (no Cityscapes equivalent) → IGNORE_INDEX
_all_coco_ids = set(range(133))
for _cid in _all_coco_ids - set(COCO_TO_CS):
    COCO_TO_CS[_cid] = IGNORE_INDEX


def remap_coco_to_cs(coco_mask: np.ndarray) -> np.ndarray:
    """Vectorised remap of a COCO semantic mask to Cityscapes train IDs."""
    max_id = max(coco_mask.max(), max(COCO_TO_CS))
    lut    = np.full(max_id + 1, IGNORE_INDEX, dtype=np.uint8)
    for coco_id, cs_id in COCO_TO_CS.items():
        lut[coco_id] = cs_id
    return lut[coco_mask]

## 6 · Part A — Qualitative visualisation

For each sample we show four panels:
1. **Input image**
2. **Cityscapes EoMT** — semantic prediction (19 road-scene classes)
3. **COCO EoMT** — panoptic prediction (133 classes, instance-coloured)
4. **Ground-truth** semantic target

In [ ]:
SAMPLE_IDXS = [0, 100, 250]

for idx in SAMPLE_IDXS:
    img_sample, target = val_dataset[idx]
    target_array = cs_model.to_per_pixel_targets_semantic([target], IGNORE_INDEX)[0].numpy()

    cs_pred             = run_semantic(cs_model,   img_sample)
    coco_sem, coco_inst = run_panoptic(coco_model, img_sample)

    cs_cmap = make_colormap([cs_pred, target_array], IGNORE_INDEX)

    print(f"\nSample #{idx}  —  GT classes: {np.unique(target_array[target_array != IGNORE_INDEX])}")

    fig, axes = plt.subplots(1, 4, figsize=(22, 5))
    axes[0].imshow(img_sample.permute(1, 2, 0).numpy().astype(np.uint8))
    axes[0].set_title("Input")
    axes[1].imshow(apply_colormap(cs_pred, cs_cmap))
    axes[1].set_title("Cityscapes EoMT\n(semantic — 19 classes)")
    axes[2].imshow(colorize_panoptic(coco_sem, coco_inst))
    axes[2].set_title("COCO EoMT\n(panoptic — 133 classes)")
    axes[3].imshow(apply_colormap(target_array, cs_cmap))
    axes[3].set_title("Ground-Truth Target")
    for ax in axes:
        ax.axis("off")
    plt.suptitle(f"Sample #{idx}", fontsize=12)
    plt.tight_layout()
    plt.savefig(f"phase2_comparison_{idx}.png", dpi=120, bbox_inches="tight")
    plt.show()

## 7 · Part B — Quantitative evaluation (full validation set)

### Evaluation strategy

All three models (EoMT-Cityscapes, EoMT-COCO, and
EoMT-FineTuned) are evaluated with an **identical pipeline**:

1. **Inference** — `run_semantic` is used for all models.
   This produces a per-pixel argmax in each model's native
   class space (19 for EoMT-Cityscapes and EoMT-FineTuned,
   133 for EoMT-COCO) without any confidence thresholding,
   ensuring the same protocol for every pixel.

2. **Remapping** — COCO predictions are translated to
   Cityscapes train IDs via `remap_coco_to_cs`. COCO classes
   with no valid Cityscapes counterpart are set to
   `IGNORE_INDEX=255`. EoMT-Cityscapes and EoMT-FineTuned
   predictions are already in the 19-class Cityscapes space
   and require no remapping.

3. **Metric** — mIoU is computed from a shared 19-class
   confusion matrix. Both `pred==255` and `target==255`
   pixels are excluded from evaluation.

In [ ]:
NUM_CS_CLASSES = 19
CS_CLASS_NAMES = [
    "road", "sidewalk", "building", "wall", "fence", "pole",
    "traffic light", "traffic sign", "vegetation", "terrain", "sky",
    "person", "rider", "car", "truck", "bus", "train",
    "motorcycle", "bicycle",
]


def _update_conf_mat(conf_mat, pred, target):
    mask   = (target != IGNORE_INDEX) & (pred != IGNORE_INDEX)
    p, t   = pred[mask].astype(np.int64), target[mask].astype(np.int64)
    valid  = (p >= 0) & (p < NUM_CS_CLASSES) & (t >= 0) & (t < NUM_CS_CLASSES)
    np.add.at(conf_mat, (t[valid], p[valid]), 1)


def _miou(conf_mat):
    tp  = np.diag(conf_mat)
    iou = np.where(
        (tp + conf_mat.sum(0) - tp + conf_mat.sum(1) - tp) > 0,
        tp / (conf_mat.sum(0) + conf_mat.sum(1) - tp),
        np.nan,
    )
    return float(np.nanmean(iou)), iou


cs_conf   = np.zeros((NUM_CS_CLASSES, NUM_CS_CLASSES), dtype=np.int64)
coco_conf = np.zeros((NUM_CS_CLASSES, NUM_CS_CLASSES), dtype=np.int64)

for img_sample, target in tqdm(val_dataset, desc="Evaluating", total=len(val_dataset)):
    target_array = cs_model.to_per_pixel_targets_semantic([target], IGNORE_INDEX)[0].numpy()

    cs_pred      = run_semantic(cs_model,   img_sample)
    coco_pred_cs = remap_coco_to_cs(run_semantic(coco_model, img_sample))

    _update_conf_mat(cs_conf,   cs_pred,      target_array)
    _update_conf_mat(coco_conf, coco_pred_cs, target_array)
ft_conf = np.zeros((NUM_CS_CLASSES, NUM_CS_CLASSES), dtype=np.int64)

#fine-tuned evaluation (comment if you want standard inference)
for img, target in tqdm(val_dataset, desc="Fine-tuned eval"):
    target_array = cs_model.to_per_pixel_targets_semantic([target], IGNORE_INDEX)[0].numpy()
    ft_pred = run_semantic(ft_model, img)
    _update_conf_mat(ft_conf, ft_pred, target_array)



In [ ]:
cs_miou,   cs_iou   = _miou(cs_conf)
coco_miou, coco_iou = _miou(coco_conf)

#fine-tuned miou
ft_miou, ft_iou = _miou(ft_conf)


col_w = max(len(n) for n in CS_CLASS_NAMES)
header = f"{'ID':>3}  {'Class':<{col_w}}  {'CS IoU':>8}  {'COCO→CS IoU':>12}  {'Fine-tuned IoU':>15}"
print(header)
print("─" * len(header))
for c, name in enumerate(CS_CLASS_NAMES):
    cs_v  = f"{cs_iou[c]:.4f}"  if not np.isnan(cs_iou[c])  else "  N/A  "
    coco_v = f"{coco_iou[c]:.4f}" if not np.isnan(coco_iou[c]) else "  N/A  "

    #finetuned model
    ft_v  = f"{ft_iou[c]:.4f}"  if not np.isnan(ft_iou[c])  else "  N/A  "
    print(f"{c:>3}  {name:<{col_w}}  {cs_v:>8}  {coco_v:>12}  {ft_v:>15}")

print(f"\nmIoU — CityScapes: {cs_miou:.4f} | COCO→CS: {coco_miou:.4f} | COCO Fine-tuned: {ft_miou:.4f}")